# Classification

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 5/7

Predicting *which category* instead of *how much* — and discovering that "accuracy"
alone can flat-out lie to you.

## 🎯 Learning Objectives

- Frame problems as binary classification using a real medical dataset.
- Train `LogisticRegression` and inspect `predict_proba` probabilities.
- Explain the default 0.5 decision threshold and shift it deliberately.
- Read a confusion matrix: TP / FP / FN / TN.
- Derive accuracy, precision, recall, F1 by hand, then verify with `classification_report`.
- Demonstrate why accuracy misleads on imbalanced data (the 95% fraud problem).
- Compare four classifiers fairly with cross-validation and peek at tree overfitting.

## 1. From Amounts to Categories

Regression answers *"how many lakh?"*; classification answers *"which one?"* — spam or
ham, benign or malignant, churn or stay. The output becomes a **class label**, and the
error story changes completely: being wrong has *direction*, and some wrong directions
cost far more than others. We'll use the Wisconsin Breast Cancer dataset: 569 tumour
samples, 30 numeric features from a fine-needle scan, label = malignant/benign.

**Syntax:**
```python
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)                  # labels y are classes, not numbers
clf.predict(X_test)                        # hard labels: 0 / 1
clf.predict_proba(X_test)                  # soft confidence: [[P(class0), P(class1)], ...]
```

In [ ]:
# Meet the data
import pandas as pd
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name="label")

print("Shape:", X.shape)
print("Classes:", dict(zip(cancer.target_names, [int((y == i).sum()) for i in range(2)])))
print("(sklearn encodes malignant=0, benign=1)")
print("\nFirst three features of one tumour:")
print(X.iloc[0, :3].to_dict())

## 2. Training a Classifier

`LogisticRegression` despite its name is a **classifier**: it draws a linear boundary,
then converts distance-to-boundary into a probability via the sigmoid curve.

In [ ]:
# Fit, predict, score
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=10000).fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"test accuracy: {clf.score(X_test, y_test):.3f}")
print("first 10 predictions:", y_pred[:10])
print("first 10 true labels:", y_test[:10])

## 3. Probabilities and the Decision Threshold

`predict` hides a two-step process: first compute a probability, then apply a rule —

```text
predict tumour as MALIGNANT  if  P(malignant) >= 0.5   (the default threshold)
```

That 0.5 is a *policy choice*, not a law of nature. In screening, a missed cancer
(false negative) costs vastly more than an extra biopsy (false positive), so clinics
deliberately **lower** the threshold.

> 🔍 **Under the Hood:** internally the model computes a raw score
> $z = w_1x_1 + \dots + w_dx_d + b$ (`decision_function` returns exactly this), then
> squashes it through the sigmoid $\sigma(z) = 1/(1+e^{-z})$, which maps any real
> number into (0, 1) to serve as a probability. Moving the threshold just slides a
> horizontal cut across this S-curve: lower cut → more positives caught, more false
> alarms too. There is no retraining involved — threshold tuning is free.

In [ ]:
# Probabilities first, labels second - and the threshold is OURS to choose
import numpy as np
import pandas as pd

proba_malignant = clf.predict_proba(X_test)[:, 0]     # column 0 = P(malignant)

preview = pd.DataFrame({
    "P(malignant)": proba_malignant[:8].round(3),
    "pred @0.5": np.where(proba_malignant[:8] >= 0.5, "malignant", "benign"),
})
print(preview.to_string(index=False))

for t in (0.5, 0.3):
    pred = np.where(proba_malignant >= t, 0, 1)       # 0 = malignant call
    flagged = int((pred == 0).sum())
    print(f"\nthreshold {t}: {flagged}/{len(pred)} tumours flagged malignant")

In [ ]:
# Lowering the threshold catches more true cancers (recall up) at a price (more alarms)
from sklearn.metrics import recall_score

for t in (0.5, 0.3):
    pred = np.where(proba_malignant >= t, 0, 1)
    rec = recall_score(y_test, pred, pos_label=0)     # recall for the malignant class
    print(f"threshold {t} -> malignant recall: {rec:.3f}")

## 4. The Confusion Matrix: Where Errors Live

Accuracy compresses everything into one number. The confusion matrix shows the
*shape* of the mistakes:

| | Predicted malignant | Predicted benign |
|---|---|---|
| **Actually malignant** | TP (caught!) | FN (missed — dangerous) |
| **Actually benign** | FP (false alarm) | TN (correct all-clear) |

**Syntax:**
```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot()
```

In [ ]:
# Draw it, then unpack the four cells
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = ["malignant", "benign"]
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"])
disp.plot(cmap="Blues", colorbar=False)
plt.title("Logistic regression on held-out tumours")
plt.show()

tn, fp, fn, tp = cm.ravel()   # NOTE: ravel order follows labels=[0, 1]
print(f"TP (malignant, caught)      : {tp}")
print(f"FN (malignant, MISSED)      : {fn}")
print(f"FP (benign, false alarm)    : {fp}")
print(f"TN (benign, correct all-clear): {tn}")

## 5. Accuracy, Precision, Recall, F1 — and When Each Matters

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad
\text{precision} = \frac{TP}{TP + FP}$$

$$\text{recall} = \frac{TP}{TP + FN} \qquad
F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

| Metric | Question it answers | Star player when |
|---|---|---|
| Accuracy | How often are we right overall? | Classes balanced, errors equally costly |
| Precision | Of our alarms, how many were real? | Spam filter (user tolerance for false alarms is low) |
| Recall | Of the real cases, how many did we catch? | **Cancer screening**, fraud, safety inspections |
| F1 | Harmonic balance of precision & recall | Imbalanced classes, no clear cost asymmetry |

The cancer argument in one sentence: a test that misses 2 cancers out of 60 has
*excellent precision* and still fails its patients — so we optimise **recall**,
accepting extra false alarms.

In [ ]:
# All four BY HAND from the confusion matrix cells
acc  = (tp + tn) / cm.sum()
prec = tp / (tp + fp)
rec  = tp / (tp + fn)
f1   = 2 * prec * rec / (prec + rec)

print(f"accuracy : {acc:.3f}   ({tp + tn} of {cm.sum()} calls correct)")
print(f"precision: {prec:.3f}   (when we shout 'malignant', how often right?)")
print(f"recall   : {rec:.3f}   (of all true malignants, share caught)")
print(f"F1       : {f1:.3f}")

In [ ]:
# One call produces the whole report card, per class
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

## 6. ⚠️ Accuracy Lies: The 95/5 Trap

Craft a fraud-style world where only 5% of transactions are fraudulent. A model
that predicts **"legit" for absolutely everything** — learning nothing at all —
scores ~95% accuracy. It also catches exactly zero fraudsters.

In [ ]:
# Build the imbalanced world and meet the do-nothing model
import numpy as np
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X_imb, y_imb = make_classification(
    n_samples=2000, n_features=10, weights=[0.95, 0.05],
    flip_y=0, random_state=42)

print(f"actual fraud rate: {y_imb.mean():.1%}")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb)

lazy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)   # 'always legit'
y_lazy = lazy.predict(X_te)

print(f"do-nothing model accuracy: {accuracy_score(y_te, y_lazy):.3f}")
print("\nIts full report:")
print(classification_report(y_te, y_lazy, target_names=["legit", "fraud"], zero_division=0))

Read that again: **94–95% accuracy, 0% recall on fraud.** Every fraudulent
transaction sails through, yet the scoreboard glows green. Whenever someone brags
about an accuracy number, your first question must be: *"what was the class
balance?"* Your second: *"what was the recall on the rare class?"*

(Real fixes exist — `class_weight="balanced"`, resampling, threshold tuning —
but they only matter once you measure the right thing.)

## 7. Four Classifiers Enter, One Leaderboard Leaves

Same folds, same data, same metric — lesson 03's fair-fight rule applied to a real
model zoo.

In [ ]:
# Cross-validated comparison on the cancer dataset
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

zoo = {
    "LogisticRegression": LogisticRegression(max_iter=10000),
    "KNN (k=7)":          KNeighborsClassifier(n_neighbors=7),
    "DecisionTree":       DecisionTreeClassifier(random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in zoo.items():
    s = cross_val_score(model, X, y, cv=5)
    rows.append({"model": name, "mean_acc": round(s.mean(), 3), "std": round(s.std(), 3)})

board = pd.DataFrame(rows).sort_values("mean_acc", ascending=False)
print(board.to_string(index=False))

## 8. Peeking Inside a Tree: `max_depth` and Overfitting

An unpruned decision tree keeps splitting until every leaf is pure — memorising noise.
Cap the depth and you cap the memorisation. Watch train and test accuracy race as the
tree grows (this chart returns, bigger, as lesson 07's main event).

In [ ]:
# Grow trees of increasing depth; track both exams
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

depths = range(1, 11)
train_scores, test_scores = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_scores.append(t.score(X_train, y_train))
    test_scores.append(t.score(X_test, y_test))

best_d = list(depths)[int(np.argmax(test_scores))]
plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores, marker="o", color="#ff7f0e", label="train")
plt.plot(depths, test_scores, marker="o", color="#1f77b4", label="test")
plt.axvline(best_d, color="#7f7f7f", linestyle="--", linewidth=1,
            label=f"best test depth ({best_d})")
plt.title("Deeper trees memorise more - test accuracy pays for it")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(pd.DataFrame({"depth": depths,
                    "train": np.round(train_scores, 3),
                    "test": np.round(test_scores, 3)}).to_string(index=False))

Train accuracy climbs monotonically toward 1.00 while test accuracy peaks early
and sags. The gap between the two curves is overfitting made visible — hold that
image, it anchors everything in lesson 07.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Judging a model by accuracy alone | Invisible on imbalanced data (the 95/5 trap) | Check the confusion matrix; track recall of the rare class |
| Forgetting which class is "positive" | Precision/recall silently computed for the wrong class | Set `pos_label` / pass `target_names`; read the report header |
| Leaving the default 0.5 threshold untouched | Policy mismatch: screening needs recall, spam needs precision | Tune the threshold on validation data |
| Using `predict_proba` column 0 blindly | Column order follows `classes_`, not intuition | Confirm with `model.classes_` first |
| Unbounded decision tree on small data | Memorised noise, poor generalisation | Constrain `max_depth` / `min_samples_leaf` |
| Comparing models on different splits/folds | Differences may be split luck, not skill | Same `cv`, same seed for every contender |

## 💡 Best Practices & Pro Tips

- **Choose the metric before choosing the model.** Write down which error hurts most;
  let that pick between optimising precision or recall.
- **Report the confusion matrix in reviews** — managers absorb it instantly, and it
  makes trade-offs explicit rather than buried in averages.
- **Probabilities are products.** Downstream systems (alert queues, dashboards) often
  want calibrated confidence, not hard labels; keep `predict_proba`.
- **Baseline with `DummyClassifier`** before celebrating any score — beating "always
  majority" is the entry fee, not the achievement.
- **AI-engineering relevance:** modern LLM judges and content-moderation filters are
  classifiers under the hood, evaluated with these exact precision/recall/F1 tools —
  the vocabulary transfers unchanged.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `LogisticRegression(max_iter=...)` | Linear boundary + sigmoid probabilities | `.fit(X_train, y_train)` |
| `predict_proba(X)` | Per-class confidence | `[:, 1]` = positive-class column |
| `confusion_matrix(y_true, y_pred)` | TP/FP/FN/TN counts | `.ravel()` unpacks them |
| `ConfusionMatrixDisplay(...).plot()` | Visual error map | Label axes with class names |
| `classification_report` | Precision/recall/F1 per class | Pass `target_names` for readable rows |
| `DummyClassifier(strategy="most_frequent")` | Do-nothing baseline | Exposes accuracy's blind spot |
| `cross_val_score(model, X, y, cv=5)` | Fair multi-model comparison | Same folds for all contenders |

Key takeaways:
- Classification outputs probabilities plus a threshold — and the threshold is a
  business decision, not a constant.
- Precision guards against false alarms; recall guards against misses; F1 balances
  them; accuracy alone can be 95% and useless.
- Always inspect the confusion matrix and the rare class.
- Tree depth trades memorisation (train score) for generalisation (test score).

## 🔗 Next Lesson

Continue to **[06_Clustering_PCA](../06_Clustering_PCA/notes.ipynb)** — put the
labels away entirely and let the data organise itself.